# Tutorial — Train YOLOv8 từ đầu tới cuối

Notebook đi từng bước train 1 model YOLOv8 phát hiện phương tiện giao thông VN. Chạy từng cell theo thứ tự.

**Dành cho ai:** biết Python cơ bản, chưa biết ML/DL/YOLO.

**Output cuối cùng:**
- 1 model `best.pt` đã train
- Report mAP/precision/recall trên val + test
- Video demo có bbox overlay
- Biểu đồ loss + confusion matrix


## Những khái niệm cần biết trước (5 phút)

### 1. Object detection
Cho ảnh → model vẽ **bounding box** (hình chữ nhật) quanh mỗi vật thể + nói nó là gì (class).

### 2. YOLO
"You Only Look Once" — họ model object detection nhanh. YOLOv8**s** = variant small (11.2M params), cân bằng tốc độ/độ chính xác.

### 3. Train / Val / Test

| Set | Dùng để | Model có "thấy" không? |
|---|---|---|
| **Train** | Dạy model học | Có — update weights từ set này |
| **Val** | Theo dõi trong train, tune hyperparam | Có — không update, chỉ nhìn số |
| **Test** | Đánh giá cuối cùng | **Không** — chỉ chạm sau khi chốt model |

### 4. Epoch / Batch / Learning rate

- **1 epoch** = model đi qua tất cả ảnh train 1 lần
- **Batch size** = số ảnh model xử lý cùng lúc trước khi update weights
- **Learning rate (lr0)** = tốc độ model "sửa mình" khi sai. Cao → nhảy loạn. Thấp → kẹt local optimum.

### 5. mAP

**mAP** (mean Average Precision) = điểm số đo "đoán đúng %". 0 → 1, càng cao càng tốt.
- **mAP@0.5**: đoán đúng nếu box dự đoán trùng ≥50% box thật (IoU ≥ 0.5)
- **mAP@0.5:0.95**: trung bình 10 ngưỡng IoU (khó hơn)

### 6. YOLO label format

Mỗi ảnh `abc.jpg` có file `abc.txt`:
```
0 0.512 0.634 0.08 0.12
1 0.730 0.550 0.14 0.09
```

Mỗi dòng = 1 bbox: `class_id center_x center_y width height` — toạ độ normalize [0,1].


## Config chung — chỉnh ở đây

Các cell dưới dùng các biến này. Chỉnh 1 lần, các bước sau tự dùng theo.


In [ ]:
from pathlib import Path

# Thư mục gốc project — đổi nếu bạn clone ra chỗ khác
PROJECT_ROOT = Path("/home/xuand/vn_traffic_ai")

# Dataset đã có sẵn (merged_v5)
DATA_ROOT = PROJECT_ROOT / "data" / "merged_v5"

# 4 class — thứ tự phải khớp với label file (.txt)
CLASS_NAMES = ["motorcycle", "car", "bus", "truck"]

# Output training
RUN_NAME = "tutorial_train_v8s"
PROJECT_DIR = PROJECT_ROOT / "runs" / "detect"

# Hyperparameters quan trọng — giải thích chi tiết ở Bước 4
EPOCHS = 40     # số lần đi qua toàn bộ data
BATCH = 16      # ảnh / 1 lần update
IMGSZ = 960     # kích thước ảnh input (phải chia hết cho 32)
LR0 = 0.005     # learning rate ban đầu
LRF = 0.01      # lr cuối = lr0 * lrf
PATIENCE = 10   # dừng sớm nếu không cải thiện N epoch
DEVICE = 0      # 0 = GPU đầu, "cpu" = CPU

print(f"Project: {PROJECT_ROOT}")
print(f"Data:    {DATA_ROOT}")
print(f"Output:  {PROJECT_DIR / RUN_NAME}")

---
## Bước 1 — Check môi trường

Kiểm tra 5 thứ bắt buộc có trước khi train:
1. Python >= 3.10
2. PyTorch cài đặt
3. CUDA (GPU) — không có GPU vẫn chạy được nhưng chậm 30×
4. Ultralytics (wrapper YOLOv8)
5. OpenCV (đọc/ghi ảnh, video)


In [ ]:
import sys, platform

print(f"Python: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
print(f"OS:     {platform.platform()}")

try:
    import torch
    print(f"\nPyTorch: {torch.__version__}")

    # Check CUDA — thư viện NVIDIA giúp torch chạy trên GPU
    if torch.cuda.is_available():
        n_gpu = torch.cuda.device_count()
        print(f"CUDA:   ✓ có {n_gpu} GPU")
        for i in range(n_gpu):
            name = torch.cuda.get_device_name(i)
            vram = torch.cuda.get_device_properties(i).total_memory / 1024**3
            print(f"  GPU {i}: {name} ({vram:.1f} GB VRAM)")
            # YOLOv8s + imgsz=960 + batch=16 cần ~6GB
            if vram < 6:
                print(f"  ⚠ VRAM < 6GB — nên giảm batch hoặc imgsz")
    else:
        print("CUDA:   ✗ không có GPU → sẽ chạy CPU (rất chậm)")
except ImportError:
    print("❌ Chưa cài PyTorch. Chạy: pip install torch")

try:
    import ultralytics
    print(f"\nUltralytics: {ultralytics.__version__}")
except ImportError:
    print("❌ Chưa cài: pip install ultralytics")

try:
    import cv2
    print(f"OpenCV: {cv2.__version__}")
except ImportError:
    print("❌ Chưa cài: pip install opencv-python")

---
## Bước 2 — Verify dataset

YOLOv8 cần layout:
```
data/merged_v5/
├── images/{train,val,test}/   (ảnh .jpg)
└── labels/{train,val,test}/   (file .txt cùng tên)
```

Cell dưới đếm số ảnh + bbox mỗi class để biết dataset gồm gì + cảnh báo nếu mất cân đối.


In [ ]:
from collections import Counter

def count_split(root, split):
    """Đếm ảnh + bbox mỗi class trong 1 split."""
    img_dir = root / "images" / split
    lbl_dir = root / "labels" / split

    # Liệt kê tất cả ảnh (.jpg + .png)
    images = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))

    class_count = Counter()
    n_with_label = n_empty = n_no_label = 0

    for img in images:
        # Mỗi "abc.jpg" phải có "abc.txt" cùng tên trong labels/
        lbl = lbl_dir / (img.stem + ".txt")
        if not lbl.exists():
            n_no_label += 1
            continue
        with open(lbl) as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        if not lines:
            n_empty += 1  # ảnh không có xe (negative sample) — YOLO chấp nhận
            continue
        n_with_label += 1
        for ln in lines:
            # Mỗi dòng: "class_id cx cy w h" — chỉ cần class_id
            class_count[int(ln.split()[0])] += 1

    return {
        "n_images": len(images),
        "with_label": n_with_label,
        "empty": n_empty,
        "no_label": n_no_label,
        "classes": dict(class_count),
    }

# Chạy cho 3 split
for split in ["train", "val", "test"]:
    s = count_split(DATA_ROOT, split)
    print(f"\n[{split.upper()}] {s['n_images']:,} ảnh "
          f"({s['with_label']:,} có bbox, {s['empty']:,} rỗng, {s['no_label']:,} thiếu label)")

    total = sum(s['classes'].values())
    for i, name in enumerate(CLASS_NAMES):
        n = s['classes'].get(i, 0)
        pct = 100*n/total if total else 0
        print(f"    {i} {name:12s}: {n:>6,} bbox ({pct:5.1f}%)")
    print(f"    Tổng: {total:,} bbox")

---
## Bước 3 — Tạo file config `data.yaml`

Ultralytics đọc file này để biết:
1. Data ở đâu (`path`, `train`, `val`, `test`)
2. Có bao nhiêu class và tên gì (`nc`, `names`)

**Thứ tự `names` quan trọng:** phải khớp với `class_id` trong file label `.txt`. Đổi thứ tự sau khi đã train = đổi label của toàn bộ dataset.


In [ ]:
import yaml

data_yaml_path = PROJECT_ROOT / "tutorial" / "notebook_data.yaml"

# names là dict {class_id: tên} — index khớp với số đầu trong .txt
yaml_content = {
    "path": str(DATA_ROOT),       # thư mục gốc dataset (absolute)
    "train": "images/train",       # relative so với `path`
    "val":   "images/val",
    "test":  "images/test",
    "nc": len(CLASS_NAMES),        # Number of Classes
    "names": {i: n for i, n in enumerate(CLASS_NAMES)},
}

# Dump ra file YAML
with open(data_yaml_path, "w") as f:
    yaml.safe_dump(yaml_content, f, sort_keys=False)

print(f"✓ Đã tạo: {data_yaml_path}\n")
print(data_yaml_path.read_text())

---
## Bước 4 — TRAIN model

Bước lâu nhất: 2-6 giờ tuỳ GPU. Nếu test flow trước, đặt `EPOCHS = 2`.

### Flow bên trong `model.train(...)`:
1. Load pretrained `yolov8s.pt` (COCO 80 class)
2. Reset detection head từ 80 → 4 class (**transfer learning**)
3. Mỗi epoch:
   - Chia ảnh thành batch (16 ảnh/batch)
   - Forward: dự đoán bbox + class
   - Loss: so sánh với ground truth (3 loss: box, cls, dfl)
   - Backward: tính gradient → update weights
   - Cuối epoch: val set → tính mAP → nếu best thì save `best.pt`
4. Hết epoch (hoặc early stop) → save confusion matrix + PR curve + loss curves

### Hyperparameters quan trọng (từ config ở đầu):
- **epochs=40**: nhỏ data (~3k ảnh) nên 40 vừa đủ, big data (>50k) chỉ cần 10-20
- **batch=16**: lớn → tận dụng GPU tốt nhưng tốn VRAM. Rule: `imgsz=960 → batch=16 trên 12GB`
- **imgsz=960**: lớn → bắt small object tốt (xe máy xa) nhưng chậm 2-4×. Phải chia hết 32
- **lr0=0.005**: fine-tune nên hạ xuống (default 0.01 quá cao, dễ quên feature COCO)
- **lrf=0.01**: lr cuối = `lr0 * lrf = 0.00005` (cosine decay)
- **patience=10**: không cải thiện 10 epoch liên tiếp → stop sớm


In [ ]:
from ultralytics import YOLO

# Bước 1: Load model với pretrained weights (ultralytics tự download lần đầu)
print("[1] Load pretrained yolov8s.pt...")
model = YOLO("yolov8s.pt")
model.info(verbose=False)  # in số params

# Bước 2: Train
# model.train() làm toàn bộ: reset head, setup optimizer, data loader, loop,
# save best/last, vẽ plots. Return DetMetrics object.
print(f"\n[2] Bắt đầu train — {EPOCHS} epoch, batch={BATCH}, imgsz={IMGSZ}")
print("    Tiến độ hiển thị bên dưới (có thể mất vài giờ)...")

results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    lr0=LR0,
    lrf=LRF,
    patience=PATIENCE,
    device=DEVICE,
    workers=8,            # process load data song song
    amp=True,             # float16 cho forward → nhanh 1.5× trên GPU mới
    seed=42,              # reproducible (ở mức độ nào đó)
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=False,       # True = overwrite; False = thêm suffix "2", "3"...
    plots=True,
    save=True,
    val=True,
    verbose=True,
)

# Bước 3: In metrics cuối
print("\n✓ XONG. Metrics trên val (epoch cuối):")
print(f"  mAP@0.5:      {results.box.map50:.3f}")
print(f"  mAP@0.5:0.95: {results.box.map:.3f}")
print(f"  Precision:    {results.box.mp:.3f}")
print(f"  Recall:       {results.box.mr:.3f}")

best_pt = PROJECT_DIR / RUN_NAME / "weights" / "best.pt"
print(f"\nBest weights: {best_pt}")

---
## Bước 5 — Evaluate chi tiết

Chấm mAP + per-class breakdown trên **val** và **test**, so sánh gap.

**Lưu ý về test set:**
- Nếu bạn train nhiều config rồi mỗi lần đều eval test → thông tin test "leak" vào quyết định → số không còn fair
- Research chuẩn: eval test đúng 1 lần sau khi chốt model

### Đọc gap val vs test:
- `test ≈ val`: generalization ổn ✓
- `test < val -5%`: overfit val, hoặc test khó hơn
- `test > val +5%`: bất thường, check data leak hoặc split không đều


In [ ]:
model_best = YOLO(str(PROJECT_DIR / RUN_NAME / "weights" / "best.pt"))

def eval_split(split):
    """Chạy eval 1 split, trả metrics + in bảng per-class."""
    print(f"\n{'='*60}\n{split.upper()}\n{'='*60}")

    r = model_best.val(
        data=str(data_yaml_path),
        split=split,            # "val" hoặc "test"
        imgsz=IMGSZ,
        batch=32,                # eval không có gradient → batch lớn hơn train được
        device=DEVICE,
        plots=True,
        verbose=False,
        project=str(PROJECT_DIR),
        name=f"tutorial_eval_{split}",
        exist_ok=True,
    )

    # In bảng per-class
    print(f"\n  {'Class':<12} {'P':>8} {'R':>8} {'mAP50':>8} {'mAP50-95':>10}")
    print(f"  {'-'*50}")
    for i, name in model_best.names.items():
        p = r.box.p[i] if i < len(r.box.p) else 0
        rc = r.box.r[i] if i < len(r.box.r) else 0
        ap50 = r.box.ap50[i] if i < len(r.box.ap50) else 0
        ap = r.box.ap[i] if i < len(r.box.ap) else 0
        print(f"  {name:<12} {p:>8.3f} {rc:>8.3f} {ap50:>8.3f} {ap:>10.3f}")
    print(f"  {'-'*50}")
    print(f"  {'ALL':<12} {r.box.mp:>8.3f} {r.box.mr:>8.3f} {r.box.map50:>8.3f} {r.box.map:>10.3f}")

    return {"map50": r.box.map50, "map": r.box.map, "p": r.box.mp, "r": r.box.mr}

val_m = eval_split("val")
test_m = eval_split("test")

# So sánh gap
print(f"\n{'='*60}\nSO SÁNH VAL vs TEST\n{'='*60}")
for key, label in [("map50", "mAP@0.5"), ("map", "mAP@0.5:0.95"), ("p", "P"), ("r", "R")]:
    diff = test_m[key] - val_m[key]
    print(f"  {label:<15} val={val_m[key]:.3f} test={test_m[key]:.3f} diff={diff:+.3f}")

gap = test_m["map50"] - val_m["map50"]
if abs(gap) < 0.03:   print("\n  ✓ Val/Test gần bằng → generalization ổn")
elif gap < -0.05:     print("\n  ⚠ Test thấp hơn val > 5% → overfit val hoặc test khó hơn")
elif gap > 0.05:      print("\n  ⚠ Test cao hơn val > 5% → bất thường, check leak")

---
## Bước 6 — Inference trên ảnh / video mới

### 2 mode:
- **`model.predict()`** — detect frame-by-frame (ảnh độc lập)
- **`model.track()`** — detect + gán ID bền (xe đi qua 10 frame → cùng ID)

### Confidence / IoU threshold:
- **conf**: filter box có `conf < threshold`. Thấp = bắt nhiều (có rác). Cao = ít FP, dễ miss. Default 0.25.
- **iou**: NMS — khi 2 box IoU ≥ threshold → giữ box conf cao. Default 0.7.


In [ ]:
import cv2
from pathlib import Path

CONF = 0.25
IOU = 0.7

# Lấy 1 ảnh thử từ test set
test_imgs = list((DATA_ROOT / "images" / "test").glob("*.jpg"))
if not test_imgs:
    print("Không có ảnh test — bỏ qua")
else:
    sample = test_imgs[0]
    print(f"Predict ảnh: {sample.name}")

    results = model_best.predict(
        source=str(sample),
        imgsz=IMGSZ, conf=CONF, iou=IOU, device=DEVICE,
        verbose=False, save=False,
    )
    r = results[0]  # vì chỉ 1 ảnh

    # r.boxes chứa tất cả detection:
    #   r.boxes.xyxy — [N, 4] toạ độ (x1, y1, x2, y2) ở pixel
    #   r.boxes.conf — [N] confidence
    #   r.boxes.cls  — [N] class id
    print(f"  Phát hiện {len(r.boxes)} object(s):")
    for i in range(len(r.boxes)):
        cid = int(r.boxes.cls[i])
        conf = float(r.boxes.conf[i])
        x1,y1,x2,y2 = r.boxes.xyxy[i].tolist()
        print(f"    [{i+1}] {model_best.names[cid]:12s} conf={conf:.2f} "
              f"bbox=({x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f})")

    # r.plot() vẽ sẵn bbox + label + conf, return numpy array (BGR)
    annotated = r.plot()

    # Hiển thị inline trong notebook
    import matplotlib.pyplot as plt
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))  # BGR → RGB cho matplotlib
    plt.axis("off")
    plt.title(f"Prediction: {sample.name}")
    plt.show()

### Chạy trên video (optional)

Cell dưới chạy `model.track()` trên 1 video + xuất video output có bbox overlay. Set `VIDEO_PATH` tới file video của bạn.


In [ ]:
# Đổi đường dẫn này tới video của bạn
VIDEO_PATH = PROJECT_ROOT / "data" / "test_videos" / "raw" / "demo_traffic.mp4"
OUT_VIDEO = PROJECT_ROOT / "results" / "tutorial_inference" / "predict.mp4"
OUT_VIDEO.parent.mkdir(parents=True, exist_ok=True)

if not VIDEO_PATH.exists():
    print(f"⚠ Không có {VIDEO_PATH} — bỏ qua. Đổi VIDEO_PATH thành video của bạn.")
else:
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video: {w}x{h} @ {fps:.1f} FPS, {n} frames")

    # mp4v: play được trên desktop, không play được browser (cần H.264)
    writer = cv2.VideoWriter(str(OUT_VIDEO), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    for i in range(n):
        ret, frame = cap.read()
        if not ret: break

        # model.track() giống predict nhưng có tracking:
        #   persist=True — giữ track state qua các frame
        #   tracker=bytetrack.yaml / botsort.yaml
        results = model_best.track(
            source=frame, imgsz=IMGSZ, conf=CONF, iou=IOU, device=DEVICE,
            persist=True, tracker="bytetrack.yaml", verbose=False,
        )
        writer.write(results[0].plot())

        if (i+1) % 30 == 0:
            print(f"  [{i+1}/{n}] ({100*(i+1)/n:.1f}%)")

    cap.release(); writer.release()
    print(f"\n✓ Output: {OUT_VIDEO}")

---
## Bước 7 — Vẽ biểu đồ phân tích training

Trong lúc train, Ultralytics ghi `results.csv` (mỗi dòng = 1 epoch). Cell dưới đọc + vẽ:
1. **Loss curves** (box/cls/dfl × train/val)
2. **mAP curves** (mAP@0.5, mAP@0.5:0.95)
3. **Precision/Recall curves**
4. **Learning rate schedule** (cosine decay)

### Cách đọc:
- **Loss train** phải giảm — phẳng = lr quá thấp
- **Loss val** lệch lên sau một lúc = bắt đầu overfit → stop sớm
- **mAP val** tăng dần → plateau = model đã hội tụ
- **lr** phải giảm theo đường cong cosine


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = PROJECT_DIR / RUN_NAME / "results.csv"
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()  # ultralytics đôi khi có khoảng trắng thừa
print(f"Load {len(df)} epoch từ {csv_path}")

# 3 subplot cho 3 loss
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, loss in zip(axes, ["box_loss", "cls_loss", "dfl_loss"]):
    ax.plot(df["epoch"], df[f"train/{loss}"], label=f"train", linewidth=2)
    val_col = f"val/{loss}"
    if val_col in df.columns:
        ax.plot(df["epoch"], df[val_col], label=f"val", linewidth=2)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title(loss)
    ax.legend(); ax.grid(True, alpha=0.3)
fig.suptitle("Loss curves", fontweight="bold")
fig.tight_layout(); plt.show()

In [ ]:
# mAP + P/R
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP@0.5", linewidth=2)
ax.plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP@0.5:0.95", linewidth=2)
# Đánh dấu best epoch (= best.pt được save ở đây)
best_i = df["metrics/mAP50(B)"].idxmax()
best_e = df["epoch"].iloc[best_i]
best_m = df["metrics/mAP50(B)"].iloc[best_i]
ax.axvline(best_e, color="red", linestyle="--", alpha=0.5)
ax.annotate(f"Best mAP@0.5\n= {best_m:.3f}\n@ epoch {int(best_e)}",
            xy=(best_e, best_m), xytext=(best_e+2, best_m-0.1), fontsize=9, color="red")
ax.set_xlabel("Epoch"); ax.set_ylabel("mAP"); ax.set_ylim(0, 1)
ax.legend(); ax.grid(True, alpha=0.3); ax.set_title("mAP qua các epoch (val)")

ax = axes[1]
ax.plot(df["epoch"], df["metrics/precision(B)"], label="Precision", linewidth=2)
ax.plot(df["epoch"], df["metrics/recall(B)"], label="Recall", linewidth=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("Score"); ax.set_ylim(0, 1)
ax.legend(); ax.grid(True, alpha=0.3); ax.set_title("Precision & Recall (val)")

fig.tight_layout(); plt.show()

In [ ]:
# Learning rate schedule
fig, ax = plt.subplots(figsize=(8, 4))
for pg in ["lr/pg0", "lr/pg1", "lr/pg2"]:  # 3 param groups: weights/biases/BN
    if pg in df.columns:
        ax.plot(df["epoch"], df[pg], label=pg, linewidth=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("Learning rate"); ax.set_yscale("log")
ax.legend(); ax.grid(True, alpha=0.3); ax.set_title("LR schedule (cosine decay)")
fig.tight_layout(); plt.show()

In [ ]:
# Summary
print("="*50 + "\nTÓM TẮT TRAINING\n" + "="*50)
print(f"  Epoch chạy: {len(df)} (dừng ở epoch {int(df['epoch'].iloc[-1])})")

best_i = df["metrics/mAP50(B)"].idxmax()
print(f"\n  Best mAP@0.5:      {df['metrics/mAP50(B)'].iloc[best_i]:.3f} (epoch {int(df['epoch'].iloc[best_i])})")
print(f"  Best mAP@0.5:0.95: {df['metrics/mAP50-95(B)'].max():.3f}")
print(f"  Final precision:   {df['metrics/precision(B)'].iloc[-1]:.3f}")
print(f"  Final recall:      {df['metrics/recall(B)'].iloc[-1]:.3f}")

# Loss có giảm không?
first = df['train/box_loss'].iloc[0]
last = df['train/box_loss'].iloc[-1]
print(f"\n  Box loss: {first:.3f} → {last:.3f} ({last-first:+.3f})")
if last > first * 0.9:
    print("    ⚠ Loss gần như không giảm — check lr, data, hoặc overfit")
else:
    print("    ✓ Loss giảm đáng kể — model đã học")

---
## Xong 🎉

Bạn đã:
- Train 1 YOLOv8s từ pretrained COCO xuống data VN
- Eval val + test, đọc per-class
- Chạy inference trên ảnh + video với tracking
- Vẽ biểu đồ phân tích training

### Bước tiếp theo

**Experiment để hiểu sâu hơn:**
1. Giảm `IMGSZ = 640` → so sánh mAP (thấy motorcycle giảm?)
2. Tăng `LR0 = 0.01` → thấy loss nhảy loạn
3. Đổi `yolov8s.pt` → `yolov8n.pt` (nano) → nhanh hơn nhưng mAP thấp
4. Giảm `EPOCHS = 10` → thấy chưa hội tụ

**Áp dụng vào data của bạn:**
1. Chuẩn bị data theo format YOLO: `images/{train,val,test}` + `labels/{train,val,test}`
2. Sửa `DATA_ROOT` + `CLASS_NAMES` ở cell config
3. Chạy lại notebook từ đầu

**Nâng cao:**
- Error analysis: xem notebook `notebooks/error_analysis.ipynb`
- Cross-eval nhiều model: `scripts/eval/eval_all_models_full.py`
- Streamlit demo UI: `streamlit run ui/streamlit_app.py`

### Lưu ý về eval honest

Số mAP đo được chỉ "fair" khi:
1. Test set video không trùng train (video-aware split — xem `scripts/data_prep/audit_leak.py`)
2. Chỉ eval test đúng 1 lần sau khi chốt model
3. Train/test cùng domain → chỉ đánh giá "nội suy trong-domain", không phải "generalize ra giao thông VN"

Chi tiết: README chính của repo § "Giới hạn của bộ eval hiện tại".
